# E4 · Inyección-recuperación (v2, nulo empírico)

**Spec:** [`docs/spec_E4_v2_codex_injection_recovery.md`](../docs/spec_E4_v2_codex_injection_recovery.md)  |  **Bloque:** E · Resultado  |  **Run de este set:** `ROXs42Bb_realigned`

Inyecta señal sintética y mide el throughput de cada método, con **continuo plano no nulo medido** y el gate de nulos calculado sobre **posiciones de control** (v2).

| | |
|---|---|
| **Entrada** | Extractores reales (C2–C6) + PSF + espectros de control de producción |
| **Salida (QC/productos)** | `stages/stage_h04_qc.json`, `tables/injection_throughput_by_method.csv` |
| **Consume aguas abajo** | E3 (throughput), G1 |


## Qué hace E4 v2 y qué cambia respecto de v1

E4 **inyecta** señales sintéticas de Hα de flujo/SNR conocidos y mide cuánto **recupera** cada método → el **throughput** (recuperado / inyectado), que es el factor con el que E3 convierte un límite de flujo en un límite de Ṁ. La grilla nominal está congelada: S/N {0,1,2,3,5,7,10} × anchos {LSF, 2×LSF} × continuo {none, flat} × posiciones {real, control1, control2, control3}.

**Qué cambia en v2** ([`docs/spec_E4_v2_codex_injection_recovery.md`](../docs/spec_E4_v2_codex_injection_recovery.md), congelada 2026-07-29). La auditoría encontró tres defectos de interpretación en v1:

| Defecto de v1 | Qué hace v2 |
|---|---|
| El gate de nulos llamaba «nulo» al caso S/N=0 **en la posición real del compañero**. Ese espectro contiene el dato científico y puede contener una línea real o un residuo: no es una posición nula | La población de V2 son **solo** `control1/2/3`. Las filas de la posición real se reportan aparte como `science_position_diagnostics`, **sin poder de veto** |
| `recovered_snr` usaba el error interno del extractor con los controles desactivados; con STAT en rojo, ese número no cumple el modelo de ruido canónico | La significancia se estima con los **espectros de control de producción** del mismo método: mediana móvil de 80 Å, el mismo matched filter de E1, y FAP de rango `(1+N(null≥obs))/(N+1)` |
| El modo de continuo `flat` corría con **amplitud cero** cuando faltaba una clave de config: era un duplicado exacto de `none`, así que V5 no probaba nada | Si no está configurado, E4 **mide** el continuo en las bandas laterales [6500,6540] y [6585,6625] Å sobre los cuatro productos que preservan continuo, lo lleva a escala NORMRAD y adopta la mediana de los valores positivos. V5 falla si `flat` es cero o idéntico a `none` |

**El gate global ya no exige cero extremos.** Con `K` filas extremas de `N`, falla solo si `P(X≥K | N, p_null=0.0455) < alpha=0.01`: un test binomial unilateral, el mismo que el gate de controles de D1. Eso prueba **exceso** de falsos positivos sin fingir independencia gaussiana canal a canal.

Lo que v2 **no** toca: la grilla, los anchos, las posiciones, las perturbaciones de PSF, los extractores ni la definición de throughput.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_h04_injection.sh --run-id $RUN
```

Pesado (~22 min grid 112 casos con `h04_process_pool`).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_h04_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_h04_injection.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_h04_qc.json', RUN_ID)
nb.show(qc, keys=['spec_version', 'continuum_injection.value', 'continuum_injection.source', 'checks.v2_nulls_clean.status', 'checks.v5_continuum.status'], title='E4')


## Los chequeos del QC, en físico

E4 inyecta líneas de flujo conocido y mide cuánto sobrevive (**throughput**). Ese número es el que convierte el límite de flujo de E1 en un límite de Ṁ, así que los chequeos vigilan que la inyección se comporte como debe.

| Chequeo | ¿Qué pregunta contesta? | Si falla |
|---|---|---|
| `v1_regression` | **¿La cadena sigue dando lo mismo que el caso histórico validado?** | Algo cambió en el camino sin que nadie lo decidiera. |
| `v2_nulls_clean` | **¿Inventa recuperaciones donde no inyectamos nada?** Filas S/N=0 **en posiciones de control**, con la FAP medida contra la distribución nula empírica de los controles de producción del mismo método. Falla por *exceso* binomial de extremos, no por tener uno. | La tasa de falsos positivos no es la que E1 supone, y toda detección marginal aguas abajo queda en entredicho. |
| `v3_monotonic` | **¿Más señal inyectada da más señal recuperada?** Throughput y completitud monótonos con S/N. | Una no-monotonía es bug o estadística insuficiente: hay que investigarla ANTES de usar el número. |
| `v4_hierarchy` | **¿Los métodos se ordenan como la física manda?** Las extracciones que usan el modelo de PSF (C3/C4) deberían recuperar al menos tanto como la apertura simple (C2), con sesgo < 5% a S/N ≥ 5. | Si la apertura supera al ajuste de PSF, algo está mal en C4 — no es que la apertura sea mejor. |
| `v5_continuum` | **¿Cuánto cuesta tener continuo debajo de la línea?** Exige además que el continuo plano sea **estrictamente positivo** y que sus filas **no sean idénticas** a las de `none` — si lo fueran, el chequeo no estaría midiendo nada. | Es el número que sostiene la decisión sobre los métodos de halo (C5/C6). |


## Resultados que llevaron a la conclusión

Continuo inyectado (medido o configurado), throughput por método y las verificaciones del `stage_h04_qc.json`. Si el QC en disco es anterior a v2 la celda lo dice: su `v2` medía otra población y no es comparable con este gate.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('E4', 'stages/stage_h04_qc.json'):
        q = nb.load_qc('stages/stage_h04_qc.json', RUN_ID)
        spec = q.get('spec_version')
        print(f"spec del QC en disco : {spec}")
        if spec != 'E4_v2':
            print('   ⚠ QC anterior a la spec v2: su gate de nulos incluía la posición REAL del\n'
                  '     compañero y su continuo `flat` pudo correr con amplitud cero. Hay que\n'
                  '     re-ejecutar E4 para leerlo con la política nueva.')
        ci = q.get('continuum_injection')
        if ci:
            print(f"\ncontinuo inyectado : {ci['value']:.3g} ({ci['scale']}, fuente: {ci['source']})")
            print(f"   bandas laterales: {ci['bands_A']}")
            for m, v in (ci.get('by_method') or {}).items():
                mark = '' if v > 0 else '   <- no positivo, excluido de la mediana'
                print(f"      {m:16s} {v:+10.3f}{mark}")
        print(f"\ngrid: {q['grid']['n_injections']} inyecciones × "
              f"{len(q['grid']['methods'])} métodos ({', '.join(q['grid']['methods'])})")
        th = (q.get('throughput') or {}).get('per_method_at_snr5') or {}
        print('\nthroughput @ SNR5:')
        for m in q['grid']['methods']:
            if m in th:
                print(f"   {m:16s} {th[m]['throughput']:.3f} ± {th[m]['err']:.3f}")
        print('\nverificaciones:')
        for name, c in q['checks'].items():
            extra = f"  {c.get('failures')}" if c.get('failures') else ''
            print(f"   {name:16s} {c.get('status')}{extra}")
        v2 = q['checks'].get('v2_nulls_clean') or {}
        if 'n_rows' in v2:
            print(f"\nV2 · nulo empírico sobre {v2.get('population')}")
            print(f"   {v2['n_extreme']} filas extremas de {v2['n_rows']} "
                  f"(esperadas ~{v2['n_rows'] * v2['p_null']:.1f} con p_null={v2['p_null']})")
            print(f"   exceso binomial p = {v2['excess_p']:.3g}  vs alpha = {v2['gate_alpha']}"
                  f"   -> {v2['status'].upper()}")
            rows = [r for r in v2.get('rows', []) if r.get('empirical_fap', 1.0) < v2['p_null']]
            if rows:
                from collections import Counter
                print('   extremos por método:', dict(Counter(r['method'] for r in rows)))
        print('\nopen_issues:')
        for s in q['open_issues']:
            print('  -', s)


## Plot 1 — throughput por método (lo que E3 consume)

El throughput @ SNR5 de cada método del grid, con el veredicto de `v4_hierarchy` en el título. Los métodos y los valores se leen del QC: si el conjunto de extractores cambió (v2 corre los seis), la figura lo refleja sola.


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_h04_qc.json', RUN_ID)
    th = q['throughput']['per_method_at_snr5']; v4 = q['checks']['v4_hierarchy']
    methods = [m for m in q['grid']['methods'] if m in th]
    vals = [th[m]['throughput'] for m in methods]; errs = [th[m]['err'] for m in methods]
    canon = 'psffit'
    cols = ['tab:green' if m == canon else 'tab:blue' for m in methods]
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.bar(range(len(methods)), vals, yerr=errs, color=cols, capsize=4)
    for i, v in enumerate(vals): ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
    ax.set_xticks(range(len(methods))); ax.set_xticklabels(methods, fontsize=8.5, rotation=15)
    ax.set_ylabel('throughput @ SNR5 (recuperado/inyectado)')
    ax.set_ylim(0, max(vals) * 1.25 if vals else 1.0)
    extra = f": {v4['failures']}" if v4.get('failures') else ''
    ax.set_title(f"E4 {q.get('spec_version','?')} · throughput por método "
                 f"(v4_hierarchy={v4['status']}{extra}; verde = canónico)")
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'e4_injection'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'throughput.png', dpi=110); print('figura ->', outdir / 'throughput.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — la curva de recuperación (pendiente = throughput)

Flujo neto recuperado vs inyectado por método. **Subconjunto mostrado:** solo la variante `nominal` con `continuum_mode='none'` (la configuración base del grid; las demás variantes se resumen en el QC). Pasa por el origen y la **pendiente es el throughput**: cuanto más plana la curva, más insensible el método. *(psffit va discontinuo para distinguirlo cuando se solapa con otro.)*

El caso `continuum_mode='flat'` es el que v2 arregla: con la amplitud medida, comparar `flat` contra `none` mide de verdad lo que cuesta tener continuo debajo de la línea (eso es `v5_continuum`).


In [ ]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    rd = nb.run_dir(RUN_ID)
    q = nb.load_qc('stages/stage_h04_qc.json', RUN_ID); th = q['throughput']['per_method_at_snr5']
    d = pd.read_csv(rd / 'tables' / 'injection_throughput_by_method.csv')
    d = d[(d.variant == 'nominal') & (d.continuum_mode == 'none')]
    methods = [m for m in q['grid']['methods'] if m in th]
    fig, ax = plt.subplots(figsize=(8.5, 4.3))
    for m in methods:
        ls = '--' if m == 'psffit' else '-'
        g = d[d.method == m].groupby('injected_flux')['recovered_flux_net'].median()
        ax.plot(g.index, g.values, 'o' + ls, ms=5, label=f"{m} (T={th[m]['throughput']:.2f})")
    mx = d['injected_flux'].max(); ax.plot([0, mx], [0, mx], 'k:', lw=0.8, label='ideal 1:1 (T=1)')
    ax.set_xlabel('flujo inyectado'); ax.set_ylabel('flujo neto recuperado (mediana)')
    ax.set_title('E4 · recuperación: pendiente = throughput; aperture/ls insensibles en el borde')
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = rd / 'plots' / 'e4_injection'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'recovery.png', dpi=110); print('figura ->', outdir / 'recovery.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **La posición real del compañero no es un nulo.** Su fila S/N=0 contiene el dato científico y puede llevar línea real o residuo; se reporta como diagnóstico y no vota el gate. La población nula son solo `control1/2/3`. · [`docs/spec_E4_v2_codex_injection_recovery.md`](../docs/spec_E4_v2_codex_injection_recovery.md)
- **La significancia sale de los controles de producción**, no de `recovered_snr`: con STAT en rojo, el error interno del extractor no cumple el modelo de ruido canónico. FAP de rango sobre la distribución nula empírica de cada método. · [`docs/noise_model.md`](../docs/noise_model.md)
- **El continuo `flat` se mide** si no está configurado (bandas laterales de Hα, cuatro productos que preservan continuo, escala NORMRAD, mediana de los positivos). Antes corría a cero y V5 era vacuo. · [`docs/spec_E4_v2_codex_injection_recovery.md`](../docs/spec_E4_v2_codex_injection_recovery.md)
- **El gate es binomial, no de tolerancia cero**: falla por *exceso* de extremos (`P(X≥K|N,p_null) < 0.01`), el mismo criterio que el gate de controles de D1. No se mueve el umbral mirando el run.
- Salvedad heredada: la regresión histórica de Stage06 sigue sin pasar ('H04 no válido para E3' formalmente); los throughputs se usan con esa salvedad declarada.


## Conclusión (leer del QC, no de aquí)

**E4 v2 cambia lo que significa «nulo limpio», así que el veredicto de V2 no es comparable con el de v1.** Los números concretos —throughput por método, continuo adoptado, filas extremas y su exceso binomial— los imprime la celda de evidencia sobre el QC de este objeto; no están copiados en esta prosa a propósito, que es justo lo que hacía que un objeto heredara las cifras de otro.

- **Qué mirar primero:** `v2_nulls_clean`. Si falla, hay **exceso de falsos positivos** en posiciones de control procesadas como el objeto, y eso afecta a cómo hay que leer cualquier detección marginal aguas abajo (E1/E3). Un fallo aquí no es un detalle de bookkeeping.
- **Después, `v5_continuum`:** con la amplitud medida, `flat` ya no puede ser un duplicado de `none`; la degradación por método es la que sostiene la discusión sobre los métodos de halo (C5/C6).
- **Y `v4_hierarchy`:** los métodos con modelo de PSF deberían recuperar al menos tanto como la apertura simple. Si el orden se rompe, el sospechoso es el extractor, no la apertura.
- **Downstream:** el throughput del método canónico es el que E3 aplica para el límite de Ṁ, y D1 v4 lo usa **solo** en las bandas de línea.
